Setup + imports

In [8]:
import sys
from pathlib import Path
import pandas as pd
import plotly.express as px
import numpy as np


ROOT = Path.cwd().parent
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))


from bootcamp_data.config import make_paths
pa = make_paths(ROOT)

Load + Audit

In [9]:
df = pd.read_parquet(pa.processed / "analytics_table.parquet")


print(f"Total Rows: {len(df):,}")


print("\nColumn Data Types (First 15) ")
print(df.dtypes.head(15))


print("\nMissing Values Report")
missing_report = df.isna().sum().sort_values(ascending=False)
print(missing_report[missing_report > 0])

Total Rows: 100

Column Data Types (First 15) 
order_id               string[python]
user_id                string[python]
amount                        Float64
quantity                        Int64
created_at        datetime64[ns, UTC]
status                         object
status_clean                   object
amount__isna                     bool
quantity__isna                   bool
status_norm            string[python]
date                           object
year                          float64
month                  string[python]
dow                            object
hour                          float64
dtype: object

Missing Values Report
quantity_r           100
amount_r             100
order_id_r           100
amount_winsor         12
amount                12
amount_is_outlier     12
quantity               8
hour                   7
year                   7
created_at             7
date                   7
dow                    7
month                  7
dtype: int64


Question 1: What is the Daily Revenue?

In [10]:
daily_trend = (
    df.groupby('date', dropna=False)['amount']
    .sum()
    .reset_index()
    .rename(columns={'amount': 'revenue'})
)


fig = px.line(daily_trend, x='date', y='revenue', 
              title="Daily Revenue Trend",
              markers=True)


fig.write_image(pa.figures / "daily_revenue.png")
fig.show()


High Daily Volatility: The revenue trend shows sharp spikes followed by days with zero revenue. This suggests the business relies on specific events or "drop" days rather than a steady stream of sales.

Concentrated Earnings: A few high-value days contribute to the majority of the monthly income, making the business highly sensitive to timing.

Seasonal Momentum: There is an upward trend in frequency toward late December, likely correlating with end-of-year shopping behavior.

Caveat: Since our data has many values like "not_a_number" and "missing", those days appear as $0 on our chart. The revenue shown is technically an under-representation of our true performance.

Question 2: Which countries are the top revenue ?

In [11]:
country_rev = (
    df.groupby('country', dropna=False)['amount']
    .sum()
    .reset_index()
    .sort_values('amount', ascending=False)
    .head(10)
)


fig = px.bar(country_rev, x='country', y='amount', 
             title="Top  Countries by Revenue",
             color='country')


fig.write_image(pa.figures / "revenue_by_country.png")
fig.show()

SA Market Dominance: Saudi Arabia is our primary market, generating roughly 4x more revenue than the UAE. Our product has clearly found a stronger "Product-Market Fit" in SA.

UAE Consistency: While smaller, the UAE represents a consistent secondary market, though it currently has a lower average order value.

Regional Concentration: Our focus is currently limited to the GCC. Expanding to neighboring countries like Kuwait or Qatar could be our next growth lever.

Caveat: If any user_id in the orders table isn't found in the users table, that revenue gets lost in a "NaN" category, which might slightly skew our country totals.

Question 3: What is the Distribution of Order Statuses?

In [12]:
status_dist = df['status_clean'].value_counts().reset_index()

fig = px.pie(status_dist, names='status_clean', values='count', 
             title="Distribution of Order Statuses",
             color='status_clean')

fig.write_image(pa.figures / "status_distribution.png")
fig.show()


Dominant Success Rate: The vast majority of our transactions are successfully reaching the "Paid" state. This confirms that our primary revenue funnel is functional and that the majority of our users are completing their purchase journey.

Clear Binary Outcome: By consolidating the statuses, we can see that our business operates on a simple "Success vs. Return" model. This makes it much easier for us to track our net profit versus gross sales.

Customer Retention Focus: Now that "Refund" is a single, clear category, we can see exactly how much of our volume is being returned. This gives us a direct KPI to improve; lowering this specific slice of the pie will directly increase our bottom line.

Caveat: The confusion we have is due to the fact that "Refunded" orders still contain an amount value in the raw data. If we sum up the total revenue, we must be careful to subtract the "Refund" slice, otherwise, we will be reporting "fake" income that we don't actually keep.

Bootstrap comparison

In [70]:
def bootstrap_diff_means(a: pd.Series, b: pd.Series, *, n_boot: int = 2000, seed: int = 0) -> dict:
    rng = np.random.default_rng(seed)
    a = pd.to_numeric(a, errors="coerce").dropna().to_numpy()
    b = pd.to_numeric(b, errors="coerce").dropna().to_numpy()
    assert len(a) > 0 and len(b) > 0, "Empty group after cleaning"

    diffs = []
    for _ in range(n_boot):
        sa = rng.choice(a, size=len(a), replace=True)
        sb = rng.choice(b, size=len(b), replace=True)
        diffs.append(sa.mean() - sb.mean())
    diffs = np.array(diffs)

    return {
        "diff_mean": float(a.mean() - b.mean()),
        "ci_low": float(np.quantile(diffs, 0.025)),
        "ci_high": float(np.quantile(diffs, 0.975)),
    }

d = df.assign(is_refund=df["status_clean"].eq("refund").astype(int))
a = d.loc[d["country"].eq("SA"), "is_refund"]
b = d.loc[d["country"].eq("AE"), "is_refund"]
print("n_SA:", len(a), "n_AE:", len(b))
print(bootstrap_diff_means(a, b, n_boot=2000, seed=0))

n_SA: 76 n_AE: 24
{'diff_mean': -0.04824561403508772, 'ci_low': -0.22807017543859648, 'ci_high': 0.10526315789473684}
